**UNIVERSIDADE DE SÃO PAULO**<br>
**MBA DATA SCIENCE & ANALYTICS USP/ESALQ**<br>
**Encontro para resolução de exercícios adicionais: Árvores e Ensemble Models**<br>
**Prof. Dr. Wilson Tarantin Junior**<br>
Aluna: Luiza Batista Laquini<br>
Turma: DSA 241<br>

*coding: utf-8*

In [1]:
#%% Importar os pacotes

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [2]:
#%% Importar o banco de dados

dados = pd.read_excel('data/dados_admissao.xlsx')
# Fonte: adaptado de https://www.kaggle.com/datasets/mohansacharya/graduate-admissions
# Mohan S Acharya, Asfia Armaan, Aneeta S Antony: A Comparison of Regression Models for Prediction of Graduate Admissions, IEEE International Conference on Computational Intelligence in Data Science 2019

In [3]:
#%% Limpeza dos dados

# Remover colunas que não serão utilizadas
dados.drop(columns=['Serial No.'], inplace=True)

In [4]:
#%% Estatísticas descritivas das variáveis

# Variáveis métricas
print(dados[['GRE', 'TOEFL', 'SOP', 'LOR', 'CGPA', 'Score']].describe())

# Variáveis categóricas
print(dados['UniversityRating'].value_counts())
print(dados['Research'].value_counts())

              GRE       TOEFL         SOP        LOR        CGPA      Score
count  500.000000  500.000000  500.000000  500.00000  500.000000  500.00000
mean   316.472000  107.192000    3.374000    3.48400    8.576440   72.17400
std     11.295148    6.081868    0.991004    0.92545    0.604813   14.11404
min    290.000000   92.000000    1.000000    1.00000    6.800000   34.00000
25%    308.000000  103.000000    2.500000    3.00000    8.127500   63.00000
50%    317.000000  107.000000    3.500000    3.50000    8.560000   72.00000
75%    325.000000  112.000000    4.000000    4.00000    9.040000   82.00000
max    340.000000  120.000000    5.000000    5.00000    9.920000   97.00000
UniversityRating
3    162
2    126
4    105
5     73
1     34
Name: count, dtype: int64
Research
1    280
0    220
Name: count, dtype: int64


In [5]:
#%% Transformando variáveis explicativas categóricas em dummies

dados = pd.get_dummies(dados, 
                       columns=['UniversityRating'], 
                       drop_first=False,
                       dtype='int')

# Note que Research já é uma dummy!

In [6]:
#%% Separando as variáveis Y e X

X = dados.drop(columns=['Score'])
y = dados['Score']

#%% Separando as amostras de treino e teste

# Vamos escolher 70% das observações para treino e 30% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    random_state=100)

In [7]:
#%%######################### Random Forest ####################################
###############################################################################
#%% Iniciando o Grid Search

## Alguns hiperparâmetros do modelo

# n_estimators: qtde de árvores na floresta
# max_depth: profundidade máxima da árvore
# max_features: qtde de variáveis X consideradas na busca pelo melhor split
# min_samples_leaf: qtde mínima de observações para ser nó folha

# Vamos aplicar um Grid Search
param_grid_rf = {
    'n_estimators': [100, 500],
    'max_depth': [5, 10],
    'max_features': [3, 5, 7],
    'min_samples_leaf': [30, 50]
}

# Identificar o algoritmo em uso
rf_grid = RandomForestRegressor(random_state=100)

# Treinar os modelos para o grid search
rf_grid_model = GridSearchCV(estimator = rf_grid, 
                             param_grid = param_grid_rf,
                             scoring='neg_mean_squared_error', # Atenção à metrica de avaliação!
                             cv=5, verbose=2)

rf_grid_model.fit(X_train, y_train)

# Verificando os melhores parâmetros obtidos
rf_grid_model.best_params_

# Gerando o modelo com os melhores hiperparâmetros
rf_best = rf_grid_model.best_estimator_

# Predict do modelo
rf_grid_pred_train = rf_best.predict(X_train)
rf_grid_pred_test = rf_best.predict(X_test)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=100; total time=   0.1s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=100; total time=   0.0s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=100; total time=   0.1s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=100; total time=   0.2s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=100; total time=   0.1s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=500; total time=   0.4s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=500; total time=   0.4s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=500; total time=   0.4s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=500; total time=   0.4s
[CV] END max_depth=5, max_features=3, min_samples_leaf=30, n_estimators=50

In [8]:
#%% Importância das variáveis preditoras

rf_features = pd.DataFrame({'features':X.columns.tolist(),
                            'importance':np.round(rf_best.feature_importances_, 4)}).sort_values(by='importance', ascending=False).reset_index(drop=True)

print(rf_features)

              features  importance
0                 CGPA      0.6632
1                  GRE      0.2061
2                TOEFL      0.1040
3                  SOP      0.0126
4                  LOR      0.0085
5             Research      0.0034
6   UniversityRating_2      0.0018
7   UniversityRating_3      0.0005
8   UniversityRating_1      0.0000
9   UniversityRating_4      0.0000
10  UniversityRating_5      0.0000


In [9]:
#%% Avaliando a RF (base de treino)

mse_train_rf_grid = mean_squared_error(y_train, rf_grid_pred_train)
mae_train_rf_grid = mean_absolute_error(y_train, rf_grid_pred_train)
r2_train_rf_grid = r2_score(y_train, rf_grid_pred_train)

print("Avaliação do Modelo (Base de Treino)")
print(f"MSE: {mse_train_rf_grid:.1f}")
print(f"RMSE: {np.sqrt(mse_train_rf_grid):.1f}")
print(f"MAE: {mae_train_rf_grid:.1f}")
print(f"R²: {r2_train_rf_grid:.1%}")

Avaliação do Modelo (Base de Treino)
MSE: 39.7
RMSE: 6.3
MAE: 4.5
R²: 79.7%


In [10]:
#%% Avaliando a RF (base de teste)

mse_test_rf_grid = mean_squared_error(y_test, rf_grid_pred_test)
mae_test_rf_grid = mean_absolute_error(y_test, rf_grid_pred_test)
r2_test_rf_grid = r2_score(y_test, rf_grid_pred_test)

print("Avaliação do Modelo (Base de Teste)")
print(f"MSE: {mse_test_rf_grid:.1f}")
print(f"RMSE: {np.sqrt(mse_test_rf_grid):.1f}")
print(f"MAE: {mae_test_rf_grid:.1f}")
print(f"R²: {r2_test_rf_grid:.1%}")

Avaliação do Modelo (Base de Teste)
MSE: 40.9
RMSE: 6.4
MAE: 4.8
R²: 80.3%


In [11]:
#%%######################### XGBoost ##########################################
###############################################################################
#%% Iniciando o Grid Search

## Alguns hiperparâmetros do modelo
# n_estimators: qtde de árvores no modelo
# max_depth: profundidade máxima das árvores
# learning_rate: taxa de aprendizagem
# colsample_bytree: percentual de variáveis X subamostradas para cada árvore

# Vamos aplicar um Grid Search
param_grid_xgb = {
    'n_estimators': [100, 500, 700],
    'max_depth': [3, 5],
    'learning_rate': [0.001, 0.01, 0.1],
    'colsample_bytree': [0.5, 0.8],
}

# Identificar o algoritmo em uso
xgb_grid = XGBRegressor(random_state=100)

# Treinar os modelos para o grid search
xgb_grid_model = GridSearchCV(estimator = xgb_grid, 
                             param_grid = param_grid_xgb,
                             scoring='neg_mean_squared_error', # Atenção à metrica de avaliação!
                             cv=5, verbose=2)

xgb_grid_model.fit(X_train, y_train)

# Verificando os melhores parâmetros obtidos
xgb_grid_model.best_params_

# Gerando o modelo com os melhores hiperparâmetros
xgb_best = xgb_grid_model.best_estimator_

# Predict do modelo
xgb_grid_pred_train = xgb_best.predict(X_train)
xgb_grid_pred_test = xgb_best.predict(X_test)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=100; total time=   0.0s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=100; total time=   0.0s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=100; total time=   0.0s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=100; total time=   0.0s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=100; total time=   0.0s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=500; total time=   0.2s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=500; total time=   0.2s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=500; total time=   0.2s
[CV] END colsample_bytree=0.5, learning_rate=0.001, max_depth=3, n_estimators=500; total time=   0.2s
[CV] END colsample_b

In [12]:
#%% Importância das variáveis preditoras

xgb_features = pd.DataFrame({'features':X.columns.tolist(),
                             'importance':np.round(xgb_best.feature_importances_, 4)}).sort_values(by='importance', ascending=False).reset_index(drop=True)

print(xgb_features)

              features  importance
0                 CGPA      0.2788
1                  GRE      0.2204
2                TOEFL      0.1505
3                  SOP      0.1101
4             Research      0.0645
5                  LOR      0.0534
6   UniversityRating_5      0.0333
7   UniversityRating_2      0.0325
8   UniversityRating_3      0.0270
9   UniversityRating_4      0.0150
10  UniversityRating_1      0.0145


In [13]:
#%% Avaliando o XGB (base de treino)

mse_train_xgb_grid = mean_squared_error(y_train, xgb_grid_pred_train)
mae_train_xgb_grid = mean_absolute_error(y_train, xgb_grid_pred_train)
r2_train_xgb_grid = r2_score(y_train, xgb_grid_pred_train)

print("Avaliação do Modelo (Base de Treino)")
print(f"MSE: {mse_train_xgb_grid:.1f}")
print(f"RMSE: {np.sqrt(mse_train_xgb_grid):.1f}")
print(f"MAE: {mae_train_xgb_grid:.1f}")
print(f"R²: {r2_train_xgb_grid:.1%}")

Avaliação do Modelo (Base de Treino)
MSE: 24.4
RMSE: 4.9
MAE: 3.5
R²: 87.5%


In [14]:
#%% Avaliando o XGB (base de teste)

mse_test_xgb_grid = mean_squared_error(y_test, xgb_grid_pred_test)
mae_test_xgb_grid = mean_absolute_error(y_test, xgb_grid_pred_test)
r2_test_xgb_grid = r2_score(y_test, xgb_grid_pred_test)

print("Avaliação do Modelo (Base de Teste)")
print(f"MSE: {mse_test_xgb_grid:.1f}")
print(f"RMSE: {np.sqrt(mse_test_xgb_grid):.1f}")
print(f"MAE: {mae_test_xgb_grid:.1f}")
print(f"R²: {r2_test_xgb_grid:.1%}")

#%% Fim!

Avaliação do Modelo (Base de Teste)
MSE: 34.4
RMSE: 5.9
MAE: 4.3
R²: 83.4%
